##### Human-Adipose-Depot-DIA (BMI-protein correlation analysis)
##### Yue (Winnie) Wen, Alex Zelter, Michael Riffle, Nina Isoherranen
##### Department of Pharmaceutics, Department of Genome Science, University of Washington-Seattle
##### 08/17/2025

In [1]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sbn

##### I. Import Data File & Data Cleaning

In [3]:
# Read data file
cleaned_protein_peak_area = pd.read_csv(r"figure-6-7-8-9-data.csv")
metadata = pd.read_csv(r"figure-7c-9-metadata.csv")

##### II. BMI-Protein Correlation

In [14]:
tested_protein_list = []
stat_list_OM = []
p_list_OM = []
stat_list_SQ = []
p_list_SQ = []
for i in cleaned_protein_peak_area['protein'].unique():
    temp = pd.pivot_table(cleaned_protein_peak_area[cleaned_protein_peak_area.protein==i], values='value', index='id', columns='tissue_type')
    tested_protein_list.append(i)
    temp_set_OM = pd.concat([temp.reset_index().OM,metadata.bmi],axis=1).dropna()
    temp_set_SQ = pd.concat([temp.reset_index().SQ,metadata.bmi],axis=1).dropna()
    stat_OM,p_OM = stats.spearmanr(temp_set_OM.OM, temp_set_OM.bmi, alternative='two-sided')
    stat_SQ,p_SQ = stats.spearmanr(temp_set_SQ.SQ, temp_set_SQ.bmi, alternative='two-sided')
    stat_list_OM.append(stat_OM)
    p_list_OM.append(p_OM)
    stat_list_SQ.append(stat_SQ)
    p_list_SQ.append(p_SQ)
BMI_Correlated_Proteins = pd.DataFrame({"Protein":tested_protein_list, "OM_p_value":p_list_OM, "SQ_p_value":p_list_SQ, "OM_stats":stat_list_OM,"SQ_stats":stat_list_SQ })
BMI_Correlated_Proteins.head(3)

,Protein,OM_p_value,SQ_p_value,OM_stats,SQ_stats
0,sp|A0A075B6H7|KV37_HUMAN,0.603244,0.103664,0.102641,-0.297842
1,sp|A0A075B6H9|LV469_HUMAN,0.282333,0.181436,0.210483,-0.246421
2,sp|A0A075B6I0|LV861_HUMAN,0.010458,0.776908,0.475982,-0.053035


In [15]:
# Multiple hypothesis correction
BMI_Correlated_Proteins["Adjusted_P_Value_OM"] = stats.false_discovery_control(BMI_Correlated_Proteins["OM_p_value"])
BMI_Correlated_Proteins["Adjusted_P_Value_SQ"] = stats.false_discovery_control(BMI_Correlated_Proteins["SQ_p_value"])
BMI_Correlated_Proteins.head(3)

,Protein,OM_p_value,SQ_p_value,OM_stats,SQ_stats,Adjusted_P_Value_OM,Adjusted_P_Value_SQ
0,sp|A0A075B6H7|KV37_HUMAN,0.603244,0.103664,0.102641,-0.297842,0.857515,0.373767
1,sp|A0A075B6H9|LV469_HUMAN,0.282333,0.181436,0.210483,-0.246421,0.660290,0.468629
2,sp|A0A075B6I0|LV861_HUMAN,0.010458,0.776908,0.475982,-0.053035,0.222079,0.900753


In [16]:
# Number of proteins have correlation with BMI in OM
len(BMI_Correlated_Proteins[BMI_Correlated_Proteins["Adjusted_P_Value_OM"]<0.05].Protein.unique())

19

In [17]:
# Number of proteins have positive correlation with BMI in OM
len(BMI_Correlated_Proteins[(BMI_Correlated_Proteins["Adjusted_P_Value_OM"]<0.05)&(BMI_Correlated_Proteins["OM_stats"]>0)].Protein.unique())

8

In [18]:
# Number of proteins have negative correlation with BMI in OM
len(BMI_Correlated_Proteins[(BMI_Correlated_Proteins["Adjusted_P_Value_OM"]<0.05)&(BMI_Correlated_Proteins["OM_stats"]<0)].Protein.unique())

11

In [19]:
# Number of proteins have correlation with BMI in SQ
len(BMI_Correlated_Proteins[BMI_Correlated_Proteins["Adjusted_P_Value_SQ"]<0.05].Protein.unique())

103

In [20]:
# Number of proteins have positive correlation with BMI in SQ
len(BMI_Correlated_Proteins[(BMI_Correlated_Proteins["Adjusted_P_Value_SQ"]<0.05)&(BMI_Correlated_Proteins["SQ_stats"]>0)].Protein.unique())

55

In [21]:
# Number of proteins have negative correlation with BMI in SQ
len(BMI_Correlated_Proteins[(BMI_Correlated_Proteins["Adjusted_P_Value_SQ"]<0.05)&(BMI_Correlated_Proteins["SQ_stats"]<0)].Protein.unique())

48

In [24]:
OM_BMI_Correlated_Proteins = BMI_Correlated_Proteins[BMI_Correlated_Proteins["Adjusted_P_Value_OM"]<0.05].Protein.unique()
SQ_BMI_Correlated_Proteins = BMI_Correlated_Proteins[BMI_Correlated_Proteins["Adjusted_P_Value_SQ"]<0.05].Protein.unique()
list(set(SQ_BMI_Correlated_Proteins)&set(OM_BMI_Correlated_Proteins))

['sp|P53007|TXTP_HUMAN',
 'sp|P52594|AGFG1_HUMAN',
 'sp|Q15811|ITSN1_HUMAN',
 'sp|Q86WU2|LDHD_HUMAN']